In [1]:
import pandas as pd

In [2]:
from elasticsearch import Elasticsearch
es = Elasticsearch('http://localhost:9200/')
es.ping()

True

In [5]:
df = pd.read_csv('data/reviewresto_sampledata.csv')
df.head()

,name,rating,review_text
0,smoke house deli,4.3,beautiful warm and lively atmosphere
1,smoke house deli,4.3,nice italian and continental food
2,smoke house deli,4.3,proper jazz music and lounge feeling
3,smoke house deli,4.3,very subtle and minimalistic interior
4,smoke house deli,4.3,bright lightings with wall paintings


In [6]:
from sentence_transformers import SentenceTransformer
# Load https://huggingface.co/sentence-transformers/all-mpnet-base-v2
model = SentenceTransformer("all-mpnet-base-v2")

c:\Users\sadhanas\OneDrive - Synopsys, Inc\Documents\Project\ReviewResto\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
df['review_vector'] = df['review_text'].apply(lambda x: model.encode(x))
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           25 non-null     object 
 1   rating         25 non-null     float64
 2   review_text    25 non-null     object 
 3   review_vector  25 non-null     object 
dtypes: float64(1), object(3)
memory usage: 932.0+ bytes


In [8]:
indexMapping = {
    "properties":{
        "name":{
            "type":"text"
        },
        "rating":{
            "type":"float"
        },
        "review_text":{
            "type":"text"
        },
        "review_vector":{
            "type":"dense_vector",
            "dims": 768,
            "index" :True,
            "similarity": "l2_norm"
        }
    }
}
es.indices.create(index="reviews", mappings=indexMapping)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'reviews'})

In [10]:
i=0
records = df.to_dict("records")
for record in records:
    es.index(
    index="reviews",
    id=i,
    document=record)
    i+=1

In [12]:
res = es.get(index="reviews", id=2)
res['_source']

{'name': 'smoke house deli',
 'rating': 4.3,
 'review_text': 'proper jazz music and lounge feeling',
 'review_vector': [-0.02113695628941059,
  -0.022178297862410545,
  0.006296934559941292,
  0.005235439632087946,
  -0.022904759272933006,
  0.023940587416291237,
  -0.07568914443254471,
  0.033859897404909134,
  -0.013449537567794323,
  0.02794942632317543,
  -0.12963181734085083,
  0.018030069768428802,
  0.018601620569825172,
  -0.014656023122370243,
  0.0034799950663000345,
  0.060109786689281464,
  0.004941381514072418,
  0.02589871548116207,
  0.05205940455198288,
  0.01336156390607357,
  -0.03886100649833679,
  -0.013325709849596024,
  0.017534328624606133,
  -0.021975908428430557,
  0.002782283816486597,
  -0.02053511142730713,
  -0.014531951397657394,
  0.04011135548353195,
  0.0017570992931723595,
  0.06072600930929184,
  0.0517396554350853,
  0.042064592242240906,
  -0.009428712539374828,
  -0.016849081963300705,
  1.2290370250411797e-06,
  0.06192256882786751,
  0.0304002985

In [13]:
es.count(index='reviews')


ObjectApiResponse({'count': 25, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}})

In [19]:
inputkey = "suggest a party place"
vector = model.encode(inputkey)

query = {
    'field': "review_vector",
    'query_vector': vector,
    'k':3,
    'num_candidates':25
}

res = es.knn_search(index='reviews', knn=query, source=['name','rating','review_text'])
res['hits']['hits']

C:\Users\sadhanas\AppData\Local\Temp\ipykernel_24212\1419417001.py:11: GeneralAvailabilityWarning: This API is in technical preview and may be changed or removed in a future release. Elastic will work to fix any issues, but features in technical preview are not subject to the support SLA of official GA features.
  res = es.knn_search(index='reviews', knn=query, source=['name','rating','review_text'])
C:\Users\sadhanas\AppData\Local\Temp\ipykernel_24212\1419417001.py:11: ElasticsearchWarning: The kNN search API has been replaced by the `knn` option in the search API.
  res = es.knn_search(index='reviews', knn=query, source=['name','rating','review_text'])


[{'_index': 'reviews',
  '_id': '19',
  '_score': 0.4991005,
  '_source': {'name': 'secret story',
   'rating': 3.7,
   'review_text': 'enjoyed with friends and family'}},
 {'_index': 'reviews',
  '_id': '5',
  '_score': 0.492602,
  '_source': {'name': 'chianti',
   'rating': 4.4,
   'review_text': 'we had a party '}},
 {'_index': 'reviews',
  '_id': '0',
  '_score': 0.4701976,
  '_source': {'name': 'smoke house deli',
   'rating': 4.3,
   'review_text': 'beautiful warm and lively atmosphere'}}]